# LegalIR Kaggle Final Production Trainer
**Pinned Runtime Commit:** `a4ef5d09988472d47c6163c10e66234bed7c83a2`

Production execution:
- Verify exact runtime SHA and canonical dataset identity
- Verify production bundle fingerprints
- Train one final BGE LoRA adapter on all 7,000 queries (effective batch 16)
- Rerank public candidates using frozen fusion & top-5 selector
- Validate strict submission criteria and package `submission.zip`


In [ ]:
# K0: Hardware & Environment Preflight
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'CUDA required!'
print(f'CUDA Devices: {torch.cuda.device_count()}')


In [ ]:
# K0.1: Clone and checkout pinned approved runtime (Zero torch reinstall)
import subprocess, sys
from pathlib import Path
!git clone https://github.com/silent9669/LegalIR.git /kaggle/working/LegalIR
%cd /kaggle/working/LegalIR
!git checkout a4ef5d09988472d47c6163c10e66234bed7c83a2

# Check and install minimal dependencies (zero PyTorch reinstallation)
required_pkgs = []
for mod, pkg in [('lightgbm', 'lightgbm'), ('sentencepiece', 'sentencepiece'), ('bm25s', 'bm25s'), ('pyvi', 'pyvi'), ('peft', 'peft'), ('accelerate', 'accelerate'), ('faiss', 'faiss-cpu'), ('psutil', 'psutil')]:
    try:
        __import__(mod)
    except ImportError:
        required_pkgs.append(pkg)
if required_pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-script-location'] + required_pkgs, check=True)


In [ ]:
# K1 - K9: Execute Kaggle Final Production Runner
!python scripts/run_kaggle_final.py \
    --dataset-dir /kaggle/input/task1-canonical-v2 \
    --bundle-dir /kaggle/input/legalir-production-bundle \
    --output-dir /kaggle/working


In [ ]:
# Verify output artifact
import os
assert os.path.exists('/kaggle/working/submission.zip'), 'submission.zip missing!'
print('[+] submission.zip successfully generated and ready for competition submission.')
